# ETL Bronze - CPTEC Obs (MERGE + SAMeT)

Carga los Parquet diarios de `weather.raw.cptec_volume/{merge,samet}/daily/` (recortados al bounding
box de la cuenca, escritos por `Daily_CPTEC_Obs.ipynb` o por el backfill local
`notebooks_local/cptec_obs/`) en `weather.bronze.merge_precip_grid` y `weather.bronze.samet_temp_grid`.
Idempotente por `(fecha, latitude, longitude)`; **actualiza** la fila cuando llega una version mas
nueva del archivo de origen (`source_last_modified` mayor): MERGE se regenera en los primeros dias del
mes siguiente y SAMeT ~7 dias despues. Si hay ZIP en `staging/` (carga masiva del backfill local,
`sync_to_databricks.py --bundle`) los descomprime en `daily/` antes de leer. Ver Decision 033.

In [ ]:
import re
import zipfile
from datetime import date, timedelta
from pathlib import Path

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

VOLUME_ROOT = Path('/Volumes/weather/raw/cptec_volume')
STAGING_DIR = VOLUME_ROOT / 'staging'
TABLES = {'merge': 'weather.bronze.merge_precip_grid', 'samet': 'weather.bronze.samet_temp_grid'}
KEY_COLS = ['fecha', 'latitude', 'longitude']
FILE_DATE_RE = re.compile(r'^(MERGE|SAMET)_(\d{4})_(\d{2})_(\d{2})\.parquet$')

try:
    dbutils.widgets.dropdown('load_mode', 'incremental', ['full', 'incremental'])
    dbutils.widgets.text('lookback_days', '60')
    load_mode = dbutils.widgets.get('load_mode')
    lookback_days = int(dbutils.widgets.get('lookback_days'))
except Exception:
    load_mode, lookback_days = 'incremental', 60

print(f'load_mode={load_mode}, lookback_days={lookback_days}')

In [ ]:
# Staging: ZIP subidos por el backfill local (un `databricks fs cp` por archivo tardaria horas con
# ~20.000 Parquet; el ZIP sin compresion sube en minutos). Se descomprime sobre daily/ del producto
# que indica el prefijo del nombre y se borra el ZIP. Un Parquet ya existente se pisa: el ZIP trae
# la version mas reciente descargada en local.
STAGING_DIR.mkdir(parents=True, exist_ok=True)
zips = sorted(p for p in STAGING_DIR.iterdir() if p.suffix.lower() == '.zip')
for zpath in zips:
    source = zpath.name.split('_')[0].lower()
    if source not in TABLES:
        print(f'staging: {zpath.name} ignorado (prefijo desconocido)')
        continue
    dest_dir = VOLUME_ROOT / source / 'daily'
    dest_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as zf:
        names = [n for n in zf.namelist() if FILE_DATE_RE.match(Path(n).name)]
        for n in names:
            with zf.open(n) as src, open(dest_dir / Path(n).name, 'wb') as dst:
                dst.write(src.read())
    zpath.unlink()
    print(f'staging: {zpath.name} -> {len(names)} archivos en {dest_dir}')
if not zips:
    print('staging: sin ZIP pendientes')

In [ ]:
def select_files(source):
    daily_dir = VOLUME_ROOT / source / 'daily'
    if not daily_dir.exists():
        return []
    files = []
    floor = None
    if load_mode != 'full':
        floor = date.today() - timedelta(days=lookback_days)
    for p in daily_dir.iterdir():
        m = FILE_DATE_RE.match(p.name)
        if not m:
            continue
        d = date(int(m.group(2)), int(m.group(3)), int(m.group(4)))
        if floor is None or d >= floor:
            files.append(str(p))
    return sorted(files)


def load_source(source):
    table_name = TABLES[source]
    files = select_files(source)
    if not files:
        print(f'[{source}] sin Parquet para procesar ({load_mode})')
        return
    print(f'[{source}] {len(files)} archivos ({load_mode})')

    raw = spark.read.parquet(*files)
    # Un mismo (fecha, punto) puede venir en mas de un archivo solo si alguien duplico dias en el
    # Volume; se conserva la version mas nueva de origen. Lat/lon ya vienen redondeados a 3
    # decimales en Landing; el round aca es defensivo para que la clave del MERGE sea exacta.
    df = (
        raw
        .withColumn('latitude', F.round(F.col('latitude'), 3))
        .withColumn('longitude', F.round(F.col('longitude'), 3))
        .withColumn('ingestion_date', F.current_date())
        .withColumn('loaded_at', F.current_timestamp())
        .withColumn('updated_at', F.current_timestamp())
        .filter(F.col('fecha').isNotNull())
    )
    w = Window.partitionBy(*KEY_COLS).orderBy(F.col('source_last_modified').desc_nulls_last(), F.col('extracted_at').desc_nulls_last())
    df = df.withColumn('rn', F.row_number().over(w)).filter(F.col('rn') == 1).drop('rn')

    bounds = df.agg(F.min('fecha').alias('lo'), F.max('fecha').alias('hi')).first()
    target = DeltaTable.forName(spark, table_name)
    # El rango de fechas en la condicion permite podar los archivos del target (CLUSTER BY fecha).
    cond = (
        f"t.fecha >= DATE '{bounds['lo']}' AND t.fecha <= DATE '{bounds['hi']}' AND "
        't.fecha = s.fecha AND t.latitude = s.latitude AND t.longitude = s.longitude'
    )
    (
        target.alias('t').merge(df.alias('s'), cond)
        .whenMatchedUpdateAll(condition='t.source_last_modified IS NULL OR s.source_last_modified > t.source_last_modified')
        .whenNotMatchedInsertAll()
        .execute()
    )
    spark.table(table_name).agg(F.min('fecha').alias('inicio'), F.max('fecha').alias('fin'), F.count('*').alias('rows'), F.countDistinct('fecha').alias('dias')).show()


for source in ('merge', 'samet'):
    load_source(source)